# PATSTAT Atypicality (z-scores) — CPC-subclass pairs, hypergeometric (Kim et al. 2016) null

The twin of `PatentView/notebook/patent_z_score.ipynb`: for each application the pairs of its own **CPC
subclasses** (4-character symbols from `tls224_appln_cpc`; applications with >= 2 subclasses) scored against
an analytic hypergeometric null on the **cumulative** application set up to the application's **filing year**.
PatentView loops in Python over 8.5 M patents; here the same cumulative counts are window sums in DuckDB, so
the arithmetic is identical and the run is minutes on 100 M applications.

$$\mu_{\alpha\beta} = \frac{n_\alpha n_\beta}{N},\quad
\sigma^2_{\alpha\beta} = \mu_{\alpha\beta}\Bigl(1 - \frac{n_\alpha}{N}\Bigr)\Bigl(\frac{N - n_\beta}{N - 1}\Bigr),\quad
z_{\alpha\beta} = \frac{o_{\alpha\beta} - \mu_{\alpha\beta}}{\sigma_{\alpha\beta}}$$
with $n$, $o$, $N$ cumulative through the filing year (all applications of that year included, as in the
PatentView loop). $z < 0$ = atypical. Per application: median, 10th percentile and min of its pairs' $z$.

## Output
- `PATSTAT/output/patstat_z_score.parquet` — `appln_id, Z_median, Z_10pct, Z_min, n_pairs`
- `PATSTAT/output/z_score_pair.parquet` — `code_1, code_2, year, Z_score` (per subclass pair per filing year in which it occurs)

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_z_score.parquet')
ps.preflight('patstat_z_score')

META = ps.out('patstat_metadata.parquet')
PAIR_FP = ps.out('z_score_pair.parquet')
con = ps.connect()

## 1. Subclass sets and pairs

In [ ]:
%%time
# 1. each application's distinct CPC subclasses (universe, >= 2 subclasses), and its pairs
con.execute(f"""CREATE OR REPLACE TABLE subs AS
  WITH s AS (SELECT DISTINCT c.appln_id, left(regexp_replace(c.cpc_class_symbol, '\\s+', '', 'g'), 4) AS sub, m.filing_year AS year
             FROM {ps.raw('tls224')} c JOIN read_parquet('{META}') m USING (appln_id))
  SELECT * FROM s QUALIFY count(*) OVER (PARTITION BY appln_id) >= 2""")
con.execute("""CREATE OR REPLACE TABLE pairs AS
  SELECT a.appln_id, a.year, a.sub AS c1, b.sub AS c2 FROM subs a JOIN subs b USING (appln_id) WHERE a.sub < b.sub""")
print(con.execute('SELECT count(DISTINCT appln_id) AS apps_with_2plus_subclasses, count(*) AS subclass_rows FROM subs').fetchdf().to_string(index=False))
print(con.execute('SELECT count(*) AS pair_rows, count(DISTINCT (c1, c2)) AS distinct_pairs FROM pairs').fetchdf().to_string(index=False))

## 2. Cumulative hypergeometric z per (pair, year)

In [ ]:
%%time
# 2. cumulative counts: N(t) applications, n_a(t) per subclass, o_ab(t) per pair -- through year t inclusive
con.execute("""CREATE OR REPLACE TABLE N_y AS
  SELECT year, sum(n) OVER (ORDER BY year) AS N FROM (SELECT year, count(DISTINCT appln_id) AS n FROM subs GROUP BY 1)""")
con.execute("""CREATE OR REPLACE TABLE n_cy AS
  SELECT sub, year, sum(n) OVER (PARTITION BY sub ORDER BY year) AS n FROM (SELECT sub, year, count(*) AS n FROM subs GROUP BY 1, 2)""")
con.execute("""CREATE OR REPLACE TABLE o_py AS
  SELECT c1, c2, year, sum(o) OVER (PARTITION BY c1, c2 ORDER BY year) AS o FROM (SELECT c1, c2, year, count(*) AS o FROM pairs GROUP BY 1, 2, 3)""")
# z for every (pair, year) in which the pair occurs; n_a(t) looked up as-of t (a subclass may have no new
# application in t), N(t) exact.
con.execute("""CREATE OR REPLACE TABLE z AS
  SELECT p.c1 AS code_1, p.c2 AS code_2, p.year,
         CASE WHEN var > 0 THEN (p.o - mu) / sqrt(var) ELSE 0.0 END AS Z_score
  FROM (
    SELECT p.*, na.n * nb.n / Ny.N AS mu,
           (na.n * nb.n / Ny.N) * (1 - na.n / Ny.N) * ((Ny.N - nb.n) / (Ny.N - 1)) AS var
    FROM o_py p
    JOIN N_y Ny USING (year)
    ASOF JOIN n_cy na ON p.c1 = na.sub AND p.year >= na.year
    ASOF JOIN n_cy nb ON p.c2 = nb.sub AND p.year >= nb.year
  ) p""")
con.execute(f"COPY (SELECT * FROM z ORDER BY year, code_1, code_2) TO '{PAIR_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'WROTE {PAIR_FP}  ({pq.ParquetFile(PAIR_FP).metadata.num_rows:,} (pair, year) rows)')

## 3. Per-application summary

In [ ]:
%%time
# 3. per application: median / 10th percentile / min over its pairs' z at its filing year
con.execute(f"""COPY (
  SELECT p.appln_id, median(z.Z_score) AS Z_median, quantile_cont(z.Z_score, 0.1) AS Z_10pct, min(z.Z_score) AS Z_min, count(*) AS n_pairs
  FROM pairs p JOIN z ON p.c1 = z.code_1 AND p.c2 = z.code_2 AND p.year = z.year
  GROUP BY 1 ORDER BY 1
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f'WROTE {OUT_FP}  ({pq.ParquetFile(OUT_FP).metadata.num_rows:,} applications)')
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') LIMIT 8").fetchdf())
display(con.execute(f"""SELECT round(avg(Z_median),3) AS mean_Z_median, round(median(Z_median),3) AS med_Z_median, round(avg(Z_10pct),3) AS mean_Z_10pct,
  round(avg(Z_min),3) AS mean_Z_min, round(100.0*avg((Z_min < 0)::INT),1) AS pct_with_atypical_pair, round(avg(n_pairs),2) AS mean_pairs FROM read_parquet('{OUT_FP}')""").fetchdf())
con.close()